## Controlled Experiment: Huber Loss

The baseline neural network performed close to Histogram Gradient Boosting but
underpredicted many high-power observations.

This experiment keeps the architecture, data split, preprocessing, optimizer,
scheduler, batch size, and early-stopping procedure unchanged while replacing
MSE loss with Huber loss.

The goal is to determine whether changing only the regression loss improves
generalization and prediction of difficult observations.

### Hypothesis

Huber loss may provide a better balance between fitting the dominant power
consumption range and handling observations with large prediction errors.

### Experimental Rule

Only the loss function will be changed so that any difference in validation
performance can be attributed primarily to the loss function.

In [2]:
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
huber_criterion = nn.HuberLoss(delta=1.0)

In [8]:
## Load Data and Recreate the Chronological Split

from pathlib import Path

data_path = Path(
    "../data/processed/household_power_features.parquet"
)

df = pd.read_parquet(data_path)
df = df.sort_values("datetime").reset_index(drop=True)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [9]:
feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440"
]

target_col = "power_target_60"

In [10]:
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

In [11]:
# scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:
# pytorch tensors

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

In [13]:
# device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [14]:
# huber

huber_model = TabularMLP(
    input_dim=X_train_tensor.shape[1]
).to(device)

In [15]:
huber_criterion = nn.HuberLoss(
    delta=1.0
)

In [16]:
# optimizer and scheduler
huber_optimizer = torch.optim.Adam(
    huber_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

huber_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    huber_optimizer,
    mode="min",
    factor=0.5,
    patience=5
)


In [18]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    max_epochs=100,
    early_stopping_patience=10,
):

    history = {
        "train_loss": [],
        "val_loss": [],
        "learning_rate": []
    }

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    for epoch in range(max_epochs):

        epoch_start = time.time()

        # Training phase
        model.train()

        train_loss_sum = 0.0
        train_samples = 0

        for xb, yb in train_loader:

            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            predictions = model(xb)

            loss = criterion(predictions, yb)

            loss.backward()

            optimizer.step()

            batch_size_actual = xb.size(0)

            train_loss_sum += loss.item() * batch_size_actual
            train_samples += batch_size_actual

        train_loss = train_loss_sum / train_samples

        # Validation phase
        model.eval()

        val_loss_sum = 0.0
        val_samples = 0

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                predictions = model(xb)

                loss = criterion(predictions, yb)

                batch_size_actual = xb.size(0)

                val_loss_sum += loss.item() * batch_size_actual
                val_samples += batch_size_actual

        val_loss = val_loss_sum / val_samples

        # Scheduler
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["learning_rate"].append(current_lr)

        # Early stopping
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        elapsed = time.time() - epoch_start

        print(
            f"Epoch {epoch + 1:3d}/{max_epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
        )

        if epochs_without_improvement >= early_stopping_patience:

            print("\nEarly stopping triggered.")
            break

    model.load_state_dict(best_model_state)

    print(
        f"\nBest validation loss: "
        f"{best_val_loss:.6f}"
    )

    return model, history

In [20]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

batch_size = 2048

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)

max_epochs = 100
early_stopping_patience = 10

In [21]:
# train huber loss function

huber_model, huber_history = train_model(
    model=huber_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=huber_criterion,
    optimizer=huber_optimizer,
    scheduler=huber_scheduler,
    device=device,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience
)

Epoch   1/100 | Train Loss: 0.271127 | Val Loss: 0.254911 | LR: 1.00e-03 | Time: 7.4s
Epoch   2/100 | Train Loss: 0.254569 | Val Loss: 0.250483 | LR: 1.00e-03 | Time: 7.2s
Epoch   3/100 | Train Loss: 0.248617 | Val Loss: 0.249792 | LR: 1.00e-03 | Time: 7.4s
Epoch   4/100 | Train Loss: 0.244840 | Val Loss: 0.248049 | LR: 1.00e-03 | Time: 7.4s
Epoch   5/100 | Train Loss: 0.242360 | Val Loss: 0.248899 | LR: 1.00e-03 | Time: 7.2s
Epoch   6/100 | Train Loss: 0.240480 | Val Loss: 0.244900 | LR: 1.00e-03 | Time: 7.4s
Epoch   7/100 | Train Loss: 0.238946 | Val Loss: 0.246438 | LR: 1.00e-03 | Time: 7.1s
Epoch   8/100 | Train Loss: 0.237757 | Val Loss: 0.249110 | LR: 1.00e-03 | Time: 7.4s
Epoch   9/100 | Train Loss: 0.236621 | Val Loss: 0.247469 | LR: 1.00e-03 | Time: 7.4s
Epoch  10/100 | Train Loss: 0.235801 | Val Loss: 0.245365 | LR: 1.00e-03 | Time: 7.4s
Epoch  11/100 | Train Loss: 0.235066 | Val Loss: 0.247348 | LR: 1.00e-03 | Time: 7.5s
Epoch  12/100 | Train Loss: 0.234436 | Val Loss: 0.251

In [22]:
# validation evaluation

huber_model.eval()

huber_predictions = []
huber_targets = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        preds = huber_model(xb)

        huber_predictions.append(
            preds.cpu().numpy()
        )

        huber_targets.append(
            yb.cpu().numpy()
        )

huber_predictions = np.concatenate(
    huber_predictions
).ravel()

huber_targets = np.concatenate(
    huber_targets
).ravel()

In [23]:
huber_mae = mean_absolute_error(
    huber_targets,
    huber_predictions
)

huber_rmse = mean_squared_error(
    huber_targets,
    huber_predictions
) ** 0.5

huber_r2 = r2_score(
    huber_targets,
    huber_predictions
)

print(f"Huber NN MAE:  {huber_mae:.4f}")
print(f"Huber NN RMSE: {huber_rmse:.4f}")
print(f"Huber NN R²:   {huber_r2:.4f}")

Huber NN MAE:  0.5086
Huber NN RMSE: 0.8103
Huber NN R²:   0.4025


In [24]:
# compare against HGB
hgb_rmse = 0.788538

vs_hgb = (
    (hgb_rmse - huber_rmse)
    / hgb_rmse
) * 100

print(
    f"Huber NN improvement vs HGB: "
    f"{vs_hgb:.2f}%"
)

Huber NN improvement vs HGB: -2.76%


In [26]:
baseline_nn_rmse = 0.7983

vs_baseline_nn = (
    (baseline_nn_rmse - huber_rmse)
    / baseline_nn_rmse
) * 100

print(
    f"Huber NN improvement vs baseline NN: "
    f"{vs_baseline_nn:.2f}%"
)

Huber NN improvement vs baseline NN: -1.51%


In [27]:
#Changing only the loss function from MSE to Huber loss did not improve validation performance. 
#The baseline MSE-trained neural network remains the stronger neural-network configuration.